# Word Frequency

**Computational Text Analysis Pack · Module 02 — Describing a Corpus**

*Technique:* Frequency distributions; raw counts vs normalised rates

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/02-describing-corpora/02a_word_frequency.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Word frequency is the foundation everything else in this pack is built on: TF-IDF,
keyness, topic models and stylometry are all frequency with something done to it.

It is also the method most often reported badly. Two rules fix most of that:

1. **Never compare raw counts across corpora of different size.** Normalise.
2. **The commonest words are the least informative.** Expect *the* and *of*.

## The tooling

**`collections.Counter`** is the whole engine of frequency analysis — a
standard-library dictionary that counts. **matplotlib** draws the distributions.
The field's dedicated tools (**AntConc**, **#LancsBox**) wrap the same counting
in a GUI.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
from collections import Counter

def frequencies(texts):
    """Token counts across a set of documents."""
    counts = Counter()
    for t in texts:
        counts.update(tokenise(t))
    return counts

def freq_frame(counts, top=25):
    total = sum(counts.values())
    rows = [{"rank": i, "word": w, "count": n, "per_1000": n / total * 1000}
            for i, (w, n) in enumerate(counts.most_common(top), start=1)]
    return pd.DataFrame(rows)

## The frequency table

`per_1000` is the column you report. A raw count of 400 means nothing until the
reader knows whether the corpus has 10,000 words or 10 million.

In [ ]:
yor = df[df["language"] == "Yoruba"]
counts_yor = frequencies(yor["text"])

print(f"Yoruba: {len(yor):,} documents, {sum(counts_yor.values()):,} tokens, "
      f"{len(counts_yor):,} types")
print()
print(freq_frame(counts_yor, top=20).to_string(index=False,
      float_format=lambda v: f"{v:.2f}"))

## Read what came back

The top of a Yoruba frequency list is *ni, ti, tó, àwọn, wọn, pé, sí* — the
grammatical machinery of the language. Exactly as an English list gives *the, of,
and, to*.

This matters more here than it looks. **None of those Yoruba words appear in any
off-the-shelf stopword list**, because those lists were built for English. If you
apply an English stopword list to Yoruba text you remove almost nothing — and if
you assume it worked, every downstream result is wrong. Notebook 03a measures it.

In [ ]:
# The same operation, four Nigerian languages side by side.
for lang in ("Hausa", "Igbo", "Yoruba", "Nigerian Pidgin"):
    sub = df[df["language"] == lang]
    c = frequencies(sub["text"])
    top = ", ".join(w for w, _ in c.most_common(8))
    print(f"{lang:<18}{sum(c.values()):>9,} tokens   {top}")

### 🔧 Your turn 1

Swap in two languages you work with, or two `category` values within one
language, and compare their top-20 lists.

How much of each list is grammatical, and how much is subject matter? That ratio
is the reason notebook 04a exists.

## Content words: the crude fix, and its cost

Dropping the top *n* most frequent words is the language-independent way to strip
function words. It is crude — it will take real content words with it — but
unlike an English stopword list it does not silently do nothing.

In [ ]:
TOP_N_TO_DROP = 40          # ← change this

stop = {w for w, _ in counts_yor.most_common(TOP_N_TO_DROP)}
content = {w: n for w, n in counts_yor.items() if w not in stop}

print(f"dropped the {TOP_N_TO_DROP} commonest types "
      f"({sum(counts_yor[w] for w in stop) / sum(counts_yor.values()):.1%} of all tokens)")
print()
for w, n in sorted(content.items(), key=lambda kv: -kv[1])[:20]:
    print(f"   {w:<18}{n:>7,}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
words = [w for w, _ in sorted(content.items(), key=lambda kv: -kv[1])[:20]][::-1]
vals = [content[w] for w in words]
ax.barh(words, vals)
ax.set_xlabel("occurrences")
ax.set_title(f"Yoruba news: commonest content words (top {TOP_N_TO_DROP} dropped)")
plt.tight_layout(); plt.show()

### 🔧 Your turn 2

Set `TOP_N_TO_DROP` to 10, then 100, and re-run.

At 10 the list is still mostly grammar. At 100 you have started deleting words
your argument might need. There is no correct value — which is why the number
belongs in your methods section, not buried in a cell.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Every language's top 20 is dominated by function words —
typically 60–80% of the list. Subject matter only starts to show around rank
15–25, and even then it is words like *government* or *people* that are common
across every category. This is the central limitation of raw frequency: it
describes the language, not the corpus.

**Your turn 2.** The honest answer is that dropping the top *n* is a blunt
instrument and there is no principled value of *n*. That is an argument for
skipping it entirely and going to keyness (04a) instead, which needs no stopword
decision at all: it compares two corpora directly, so the words common to both
cancel out on their own. Report your *n* if you use one; better, do not need one.

</details>

## Where this stops

You can count words and normalise them. You cannot yet say whether a frequency is
*unusual*. Next: **02b — Zipf's law**, which says what "usual" looks like.